In [ ]:
import zipfile
import os

zip_file_path = '/content/mtsamples.csv.zip'
extract_dir = '/content/'

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"'{zip_file_path}' unzipped to '{extract_dir}' successfully!")

print("Contents of extraction directory:")
print(os.listdir(extract_dir))

'/content/mtsamples.csv.zip' unzipped to '/content/' successfully!
Contents of extraction directory:
['.config', 'mtsamples.csv', 'mtsamples.csv.zip', '.ipynb_checkpoints', 'sample_data']


In [ ]:
import pandas as pd

In [ ]:
file_path = '/content/mtsamples.csv'
df = pd.read_csv(file_path)
display(df.head())

,Unnamed: 0,description,medical_specialty,sample_name,transcription,keywords
0,0,A 23-year-old white female presents with comp...,Allergy / Immunology,Allergic Rhinitis,"SUBJECTIVE:, This 23-year-old white female pr...","allergy / immunology, allergic rhinitis, aller..."
1,1,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 2,"PAST MEDICAL HISTORY:, He has difficulty climb...","bariatrics, laparoscopic gastric bypass, weigh..."
2,2,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 1,"HISTORY OF PRESENT ILLNESS: , I have seen ABC ...","bariatrics, laparoscopic gastric bypass, heart..."
3,3,2-D M-Mode. Doppler.,Cardiovascular / Pulmonary,2-D Echocardiogram - 1,"2-D M-MODE: , ,1. Left atrial enlargement wit...","cardiovascular / pulmonary, 2-d m-mode, dopple..."
4,4,2-D Echocardiogram,Cardiovascular / Pulmonary,2-D Echocardiogram - 2,1. The left ventricular cavity size and wall ...,"cardiovascular / pulmonary, 2-d, doppler, echo..."


In [ ]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4999 entries, 0 to 4998
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Unnamed: 0         4999 non-null   int64 
 1   description        4999 non-null   object
 2   medical_specialty  4999 non-null   object
 3   sample_name        4999 non-null   object
 4   transcription      4966 non-null   object
 5   keywords           3931 non-null   object
dtypes: int64(1), object(5)
memory usage: 234.5+ KB
None


In [ ]:
print("\nFirst 5 rows of the DataFrame:")
display(df.head())


First 5 rows of the DataFrame:


,Unnamed: 0,description,medical_specialty,sample_name,transcription,keywords
0,0,A 23-year-old white female presents with comp...,Allergy / Immunology,Allergic Rhinitis,"SUBJECTIVE:, This 23-year-old white female pr...","allergy / immunology, allergic rhinitis, aller..."
1,1,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 2,"PAST MEDICAL HISTORY:, He has difficulty climb...","bariatrics, laparoscopic gastric bypass, weigh..."
2,2,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 1,"HISTORY OF PRESENT ILLNESS: , I have seen ABC ...","bariatrics, laparoscopic gastric bypass, heart..."
3,3,2-D M-Mode. Doppler.,Cardiovascular / Pulmonary,2-D Echocardiogram - 1,"2-D M-MODE: , ,1. Left atrial enlargement wit...","cardiovascular / pulmonary, 2-d m-mode, dopple..."
4,4,2-D Echocardiogram,Cardiovascular / Pulmonary,2-D Echocardiogram - 2,1. The left ventricular cavity size and wall ...,"cardiovascular / pulmonary, 2-d, doppler, echo..."


In [ ]:
print("Number of missing values in 'transcription' column:", df['transcription'].isnull().sum())
print("Number of missing values in 'keywords' column:", df['keywords'].isnull().sum())

df_cleaned = df.dropna(subset=['transcription']).copy()
print(f"\nDataFrame shape after dropping rows with missing transcription: {df_cleaned.shape}")

print("\nValue counts for 'medical_specialty':")
print(df_cleaned['medical_specialty'].value_counts())

Number of missing values in 'transcription' column: 33
Number of missing values in 'keywords' column: 1068

DataFrame shape after dropping rows with missing transcription: (4966, 6)

Value counts for 'medical_specialty':
medical_specialty
Surgery                          1088
Consult - History and Phy.        516
Cardiovascular / Pulmonary        371
Orthopedic                        355
Radiology                         273
General Medicine                  259
Gastroenterology                  224
Neurology                         223
SOAP / Chart / Progress Notes     166
Urology                           156
Obstetrics / Gynecology           155
Discharge Summary                 108
ENT - Otolaryngology               96
Neurosurgery                       94
Hematology - Oncology              90
Ophthalmology                      83
Nephrology                         81
Emergency Room Reports             75
Pediatrics - Neonatal              70
Pain Management                    61
P

In [ ]:
# get rid of super short/long samples
df_cleaned["word_count"] = df_cleaned["transcription"].str.split().str.len()

candidates = df_cleaned[
    df_cleaned["word_count"].between(100, 800)
].drop_duplicates(subset="transcription")

# use a mix of specialties
top_specialties = candidates["medical_specialty"].value_counts().head(10).index

# 5 from each = 50 total
sampled_df = (
    candidates[candidates["medical_specialty"].isin(top_specialties)]
    .groupby("medical_specialty", group_keys=False)
    .sample(n=5, random_state=42)
    .reset_index(drop=True)
)

print("total:", len(sampled_df))
print(sampled_df["medical_specialty"].value_counts())

sampled_df[["medical_specialty", "sample_name", "word_count", "transcription"]].head()

total: 50
medical_specialty
Consult - History and Phy.       5
General Medicine                 5
Neurology                        5
Office Notes                     5
Orthopedic                       5
Pediatrics - Neonatal            5
Radiology                        5
SOAP / Chart / Progress Notes    5
Surgery                          5
Urology                          5
Name: count, dtype: int64


,medical_specialty,sample_name,word_count,transcription
0,Consult - History and Phy.,Bariatric Consult - Surgical Weight Loss - 4,642,"HISTORY OF PRESENT ILLNESS:, Ms. A is a 55-ye..."
1,Consult - History and Phy.,Consult - Atrial Fibrillation,719,"REASON FOR CONSULTATION:, Atrial fibrillation..."
2,Consult - History and Phy.,Consult - Atrial Fibrillation - 1,339,"REASON FOR CONSULTATION: , Atrial fibrillation..."
3,Consult - History and Phy.,Allergy Evaluation Consult,638,"HISTORY: , A 34-year-old male presents today s..."
4,Consult - History and Phy.,Cardiac Consultation - 2,432,"CHIEF REASON FOR CONSULTATION:, Evaluate exer..."


In [20]:
!pip install -q -U google-genai

from google import genai
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini configured")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262.5 kB 21.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.59.0 which is incompatible.
Gemini configured


In [23]:
def simplify_text_with_gemini(text):
    prompt = (
        "Simplify the following medical text to an 8th-grade reading level. "
        "Keep all important medical information, diagnoses, and procedures accurate. "
        "Do not add information that is not in the original text.\n\n"
        f"{text}"
    )

    try:
        response = client.models.generate_content(
            model="gemini-3.5-flash-lite",
            contents=prompt
        )
        return response.text
    except Exception as e:
        print("error:", e)
        return None

In [24]:
test = simplify_text_with_gemini(sampled_df.loc[0, "transcription"])
print(test)

**HISTORY OF PRESENT ILLNESS:** 
Ms. A is a 55-year-old woman who came to the Bariatric Surgery Service to see about getting a laparoscopic Roux-en-Y gastric bypass. She says she has been overweight for about 35 years. She has tried many ways to lose weight in the past, including Weight Watchers, NutriSystem, Jenny Craig, TOPS, the cabbage diet, the grapefruit diet, Slim-Fast, Richard Simmons, and over-the-counter products, but she has not been able to keep the weight off long-term. When she came to the office, she was 5 feet 6 inches tall, weighed 285.4 pounds, and had a body mass index (BMI) of 46. She has health problems related to her obesity, which include high blood pressure and high cholesterol.

**PAST MEDICAL HISTORY:** 
She has high blood pressure, for which she takes Norvasc and Lopressor. She also has high cholesterol and takes lovastatin for it. She has depression, for which she takes citalopram. She said she had a blood clot (DVT) in the past before her hysterectomy. She 

In [25]:
import time

# run all 50
results = []

#retries failed requests and saves progress as the cell runs
for i, text in enumerate(sampled_df["transcription"]):
    simplified = None

    for attempt in range(5):
        simplified = simplify_text_with_gemini(text)

        if simplified is not None:
            break

        print(f"retrying {i + 1}...")
        time.sleep(5 * (attempt + 1))

    results.append(simplified)

    # save progress
    sampled_df.loc[i, "simplified_transcription"] = simplified
    sampled_df.to_csv("/content/mtsamples_baseline_outputs.csv", index=False)

    print(f"done {i + 1}/50")

done 1/50
done 2/50
done 3/50
done 4/50
done 5/50
done 6/50
done 7/50
done 8/50
done 9/50
done 10/50
done 11/50
done 12/50
done 13/50
done 14/50
done 15/50
done 16/50
done 17/50
done 18/50
done 19/50
done 20/50
done 21/50
done 22/50
done 23/50
done 24/50
done 25/50
done 26/50
done 27/50
done 28/50
done 29/50
done 30/50
done 31/50
done 32/50
done 33/50
done 34/50
done 35/50
done 36/50
done 37/50
done 38/50
done 39/50
done 40/50
done 41/50
done 42/50
done 43/50
done 44/50
done 45/50
error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 15, model: gemini-3.5-flash-lite\nPlease retry in 42.215161222s.', 'status': 'RESOURCE_EXHAUSTED',

In [26]:
# save results
output_file = "/content/mtsamples_baseline_outputs.csv"

sampled_df.to_csv(output_file, index=False)

print("saved:", output_file)
print("total:", len(sampled_df))

saved: /content/mtsamples_baseline_outputs.csv
total: 50


In [28]:
#ensure everything ran properly
print("successful:", sampled_df["simplified_transcription"].notna().sum())
print("failed:", sampled_df["simplified_transcription"].isna().sum())

successful: 50
failed: 0


In [29]:
from google.colab import files

files.download("/content/mtsamples_baseline_outputs.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>